In [1]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


A. EXTRACT (bobot 15%)

Baca ketiga sumber data (tugas6_transaksi.csv, tugas6_produk.json, tugas6_ulasan.csv) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan printSchema() masing-masing.

In [16]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas_6") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_t = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)
df_p = spark.read.json("tugas6_produk.json")
df_u = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("tugas6_transaksi.csv")
df_t.printSchema()
j_t = df_t.count()
print("jumlah baris : ", j_t)

print("tugas6_produk.json")
df_p.printSchema()
j_p = df_p.count()
print("jumlah baris : ", j_p)

print("tugas6_ulasan.csv")
df_u.printSchema()
j_u = df_u.count()
print("jumlah baris : ", j_u)

tugas6_transaksi.csv
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

jumlah baris :  5000
tugas6_produk.json
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

jumlah baris :  30
tugas6_ulasan.csv
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)

jumlah baris :  3500


In [10]:
df_t.show()
df_p.show()
df_u.show()

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
|     TX5|        10|           3|2026-10-01 00:00:00|
|     TX6|        26|           6|2026-10-24 00:00:00|
|     TX7|         4|           1|2026-10-03 00:00:00|
|     TX8|         7|           2|2026-10-09 00:00:00|
|     TX9|        30|           1|2026-10-27 00:00:00|
|    TX10|        20|           1|2026-10-04 00:00:00|
|    TX11|        15|           4|2026-10-01 00:00:00|
|    TX12|        29|           3|2026-10-02 00:00:00|
|    TX13|        11|           7|2026-10-24 00:00:00|
|    TX14|        14|           1|2026-10-01 00:00:00|
|    TX15|

B. TRANSFORM — Penggabungan (bobot 25%)

Gabungkan ketiga DataFrame menjadi satu (order_id sebagai kunci ke ulasan, product_id sebagai kunci ke produk). Gunakan salah satu join yang tepat dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan salah satu join yang tepat dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom total_pendapatan (unit_terjual x harga).

In [18]:
from pyspark.sql.functions import col, sum as spark_sum, avg, count

gabung1 = df_t.join(df_p, on="product_id", how="left")
gabung2 = gabung1.join(df_u, on="order_id", how="left")

answer2 = gabung2.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
answer2.show()

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|  NULL|         1500000|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|  NULL|          450000|
|     TX5|        10|           3|2026-10-01 00:00:00| 75000|  Elektronik|  Produk-10|     5|          225000|
|

C. TRANSFORM — Penanganan Data Kosong & Pengayaan (bobot 20%)

Transaksi tanpa ulasan akan memiliki rating bernilai kosong (null) setelah salah satu join yang tepat — isi nilai kosong tersebut dengan angka 0 menggunakan salah satu function, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
Tambahkan kolom ada_ulasan bernilai True/False (tidak boleh diisi manual satu satu), **sebelum** langkah na.fill()` di atas).

In [23]:
kolom_baru = answer2.withColumn("ada_ulasan", col("rating").isNotNull())

answer3 = kolom_baru.na.fill(0, subset=["rating"])
answer3.show()

# alasan kenapa 0 masuk akal untuk kasus "belum ada ulasan": karena dalam sistem rating yang melibatkan angka 1 sampai 5, angka 0 berada
# diluar jangkauan sehingga angka 0 dinilai cocok sebagai indikator bahwa rating belum diberikan. Ini juga sebagai indikator kalau rating
# masih berstatus false

+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|order_id|product_id|unit_terjual|            tanggal| harga|    kategori|nama_produk|rating|total_pendapatan|ada_ulasan|
+--------+----------+------------+-------------------+------+------------+-----------+------+----------------+----------+
|     TX0|        21|           2|2026-10-10 00:00:00| 25000|  Elektronik|  Produk-21|     1|           50000|      true|
|     TX1|         8|           6|2026-10-25 00:00:00|250000|     Fashion|   Produk-8|     0|         1500000|     false|
|     TX2|        25|           4|2026-10-13 00:00:00| 25000|  Elektronik|  Produk-25|     2|          100000|      true|
|     TX3|         3|           4|2026-10-18 00:00:00| 75000|     Fashion|   Produk-3|     5|          300000|      true|
|     TX4|        19|           6|2026-10-07 00:00:00| 75000|Rumah Tangga|  Produk-19|     0|          450000|     false|
|     TX5|        10|   

D. LOAD (bobot 25%)

Simpan hasil akhir ke HDFS dalam format Parquet, dipartisi berdasarkan kategori, ke path /user/[username]/tugas6/hasil_etl. Verifikasi dengan hdfs dfs -ls -R, lalu baca kembali dan tampilkan count()-nya sebagai bukti data tersimpan utuh.



In [26]:
!hdfs dfs -mkdir -p /home/plont/Documents/Praktikum_Big_Data/tugas/tugas_6/hasil_etl
path = "/home/plont/Documents/Praktikum_Big_Data/tugas/tugas_6/hasil_etl"

# Simpan DataFrame answer3 ke format Parquet, dipartisi berdasarkan 'kategori'
# Gunakan mode("overwrite") agar tidak error jika kamu me-run cell ini berkali-kali
answer3.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(path)

# 1. Verifikasi struktur file dan partisi di HDFS
!hdfs dfs -ls -R /home/plont/Documents/Praktikum_Big_Data/tugas/tugas_6/hasil_etl

print("-" * 50)

# 2. Baca kembali data Parquet yang sudah disimpan
df_hasil_etl = spark.read.parquet(path)

# 3. Tampilkan count() sebagai bukti
jumlah_awal = answer3.count()
jumlah_akhir = df_hasil_etl.count()

print(f"Jumlah baris sebelum disimpan : {jumlah_awal}")
print(f"Jumlah baris setelah dibaca   : {jumlah_akhir}")

if jumlah_awal == jumlah_akhir:
    print("Data tersimpan utuh")
else:
    print("data tidak sesuai")

--------------------------------------------------
Jumlah baris sebelum disimpan : 5000
Jumlah baris setelah dibaca   : 5000
Data tersimpan utuh


E. Insight Akhir (bobot 15%)

Dari data hasil ETL, tampilkan (menggunakan DataFrame API atau Spark SQL, bebas memilih): kategori produk mana yang memiliki persentase transaksi dengan ulasan (ada_ulasan = True) paling rendah? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

In [27]:
from pyspark.sql.functions import col, sum as spark_sum, count, round

# Menghitung persentase ulasan per kategori
df_persentase = df_hasil_etl.groupBy("kategori") \
    .agg(
        count("*").alias("total_transaksi"),
        # Mengubah True/False menjadi 1/0 lalu dijumlahkan
        spark_sum(col("ada_ulasan").cast("int")).alias("transaksi_ada_ulasan") 
    ) \
    .withColumn("persentase_ulasan", round((col("transaksi_ada_ulasan") / col("total_transaksi")) * 100, 2)) \
    .orderBy("persentase_ulasan")

df_persentase.show()

+------------+---------------+--------------------+-----------------+
|    kategori|total_transaksi|transaksi_ada_ulasan|persentase_ulasan|
+------------+---------------+--------------------+-----------------+
|     Makanan|            536|                 369|            68.84|
|   Kesehatan|            961|                 662|            68.89|
|     Fashion|           1035|                 726|            70.14|
|Rumah Tangga|            815|                 575|            70.55|
|  Elektronik|           1653|                1168|            70.66|
+------------+---------------+--------------------+-----------------+



Hal ini dinilai penting bagi tim marketing karena ulasan dinilai sangat berguna bagi developer agar bisa mengembangkan produknya menjadi lebih baik. Tanpa ulasan, produk tersebut lebih sulit untuk diketahui bagian kekurangannya. Minimnya rating juga terkadang membuat pembeli lain menjadi ragu akan produk tersebut sehingga developer bisa kehilangan calon pembeli